# MGS-23 : DifferentialEvolution MGS contre mealpy — l'écart du PSO se reproduit-il ?

**Navigation** : [<< MGS-22 (PSO vs mealpy)](MGS-22-MGS-vs-Mealpy.ipynb) | [Index](README.md)

**Kernel** : .NET (C#) — pont PythonNet vers mealpy dans la même exécution

***

## Introduction

MGS-22 a mesuré, sur le Sudoku en représentation continue R1 et à budget d'évaluations égalisé,
un écart net de qualité : mealpy `OriginalPSO` domine le composé MGS `ParticleSwarmOptimization`
(28,5 contre 43,5 conflits médians), alors que la fitness C# isolée est 5,7× plus rapide et le
coût par évaluation identique. L'hypothèse ouverte : **cet écart est-il une propriété systématique
des composés géométriques MGS, ou un accident du portage PSO ?**

Ce notebook confronte la paire la plus propre pour trancher : **Differential Evolution**. Les deux
implémentations suivent la même récurrence de base — mutation
`v = x_r1 + F·(x_r2 − x_r3)`, croisement binomial au taux CR, sélection gloutonne — donc un écart
de convergence y est plus difficile à attribuer à une divergence de définition. Enfants de l'Epic
#12373 (comparaison appariée MGS ↔ mealpy) : une paire, un notebook, une PR.

***

## 1. Le protocole apparié, pré-enregistré — hérité de MGS-22, non renégocié

Le protocole est celui de MGS-22 (#12302), repris intégralement pour que les paires de l'Epic
soient comparables entre elles :

- **même substrat** : grille Easy[0] de Sudoku_Easy51.txt, représentation R1 (continu + arrondi), fonction de coût = conflits totaux d'une grille pleine ;
- **budget d'évaluations égalisé** — mesuré, pas supposé : les compteurs des deux moteurs sont rapportés tels quels ;
- **4 graines nommées {0, 1, 7, 42}**, médiane + min/max, jamais un run isolé ;
- **contre-vérification croisée du coût** : le vainqueur mealpy est décodé et coûté côté C# — sans elle on compare deux fonctions de coût, pas deux moteurs ;
- **ms/éval séparé du temps total** ;
- **graines passées explicitement aux deux moteurs** : `solve(prob, seed=N)` côté mealpy (le paramètre du constructeur est silencieusement ignoré en 3.x), `ResetSeed(N)` côté MGS ;
- **déterminisme vérifié** (répétition, conflits identiques exigés) avant de publier le moindre chiffre.

**Paramètres par défaut de chaque bibliothèque, mesurés et déclarés** (c'est le protocole MGS-22 :
on compare les bibliothèques telles que leurs auteurs les livrent) :

| moteur | F (facteur d'échelle) | CR (croisement) |
|---|---|---|
| MGS `DifferentialEvolution` | **0,5** (`DefaultScaleFactor`) | **0,9** (`DefaultCrossoverRate`) |
| mealpy `OriginalDE` | **0,1** (`wf`) | **0,9** (`cr`) |

Le facteur F diffère par défaut (0,5 contre 0,1) : ce confond est déclaré ici, mesuré dans les
sorties, et l'exercice 1 le neutralise en alignant `wf` — la question « mécanique ou paramètre »
y trouve sa réponse partielle.

In [1]:
// === MGS-23 : socle commun — DLLs MGS, grille de référence, fonction de coût ===
// Même socle que MGS-22 : la représentation R1 (continu + arrondi) est le substrat du bench.
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"
using MetaGeneticSharp;
using GeneticSharp;
using System.Diagnostics;

// Grille facile Easy[0] de Sudoku_Easy51.txt — la MÊME que MGS-21/22.
public static string PuzzleLine23 = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";

public static int[,] ParsePuzzle23()
{
    var g = new int[9, 9];
    for (int i = 0; i < 81; i++) g[i / 9, i % 9] = PuzzleLine23[i] - '0';
    return g;
}

// Fonction de coût du bench : conflits totaux (lignes + colonnes + blocs) sur grille PLEINE.
// Renvoie 0 ssi résolu. Le côté Python réimplémente exactement ce comptage.
public static int CountConflicts23(int[,] g)
{
    int conflicts = 0;
    for (int i = 0; i < 9; i++)
    {
        var row = new HashSet<int>(); var col = new HashSet<int>(); var blk = new HashSet<int>();
        for (int j = 0; j < 9; j++)
        {
            if (!row.Add(g[i, j])) conflicts++;
            if (!col.Add(g[j, i])) conflicts++;
            int br = 3 * (i / 3) + j / 3, bc = 3 * (i % 3) + j % 3;
            if (!blk.Add(g[br, bc])) conflicts++;
        }
    }
    return conflicts;
}

public static int CountEmpty23(int[,] p) { int n = 0; foreach (var v in p) if (v == 0) n++; return n; }

public static List<(int r, int c)> EmptyCells23(int[,] p)
{
    var l = new List<(int, int)>();
    for (int r = 0; r < 9; r++) for (int c = 0; c < 9; c++) if (p[r, c] == 0) l.Add((r, c));
    return l;
}

// Décodage R1 : arrondi + clamp vers 1..9 sur les cellules vides, ordre de lecture.
public static int[,] DecodeR1_23(double[] genes)
{
    var Puzzle = ParsePuzzle23();
    var empties = EmptyCells23(Puzzle);
    var g = (int[,])Puzzle.Clone();
    for (int k = 0; k < empties.Count; k++)
        g[empties[k].r, empties[k].c] = Math.Max(1, Math.Min(9, (int)Math.Round(genes[k])));
    return g;
}

var Puzzle23 = ParsePuzzle23();
Console.WriteLine($"Grille de référence : {CountEmpty23(Puzzle23)} cellules vides, " +
                  $"{81 - CountEmpty23(Puzzle23)} indices fixes, {EmptyCells23(Puzzle23).Count} gènes R1.");

Grille de référence : 36 cellules vides, 45 indices fixes, 36 gènes R1.


**Lecture.** Le socle est posé, identique à MGS-22 au nom près — c'est voulu : la comparabilité
de l'Epic #12373 tient à ce que chaque paire courre sur exactement le même substrat. 36 cellules
vides = 36 gènes continus dans [1, 10), la fonction de coût compte les doublons ligne/colonne/bloc
d'une grille pleine et vaut 0 ssi résolue.

In [2]:
// === Moteur MGS : chromosome R1, fitness instrumentée, composé DifferentialEvolution ===
// DE/rand/1/bin canonique de Storn & Price (1997), composé géométrique MGS :
// mutant v = x_r1 + F*(x_r2 - x_r3), croisement binomial CR, sélection gloutonne héritée.
public class SudokuR1Chromosome23 : ChromosomeBase
{
    private const double LO = 1.0, HI = 10.0;
    public SudokuR1Chromosome23() : base(EmptyCells23(ParsePuzzle23()).Count) { CreateGenes(); }
    public override Gene GenerateGene(int index)
        => new Gene(RandomizationProvider.Current.GetDouble(LO, HI));
    public override IChromosome CreateNew() => new SudokuR1Chromosome23();
    public double[] ToGenes() { var v = new double[Length]; for (int i = 0; i < Length; i++) v[i] = (double)GetGene(i).Value; return v; }
    public int[,] ToGrid() => DecodeR1_23(ToGenes());
}

// Fitness instrumentée : chaque évaluation est comptée — le budget se mesure, il ne se suppose pas.
public class SudokuR1Fitness23 : IFitness
{
    public static int Evals;
    public double Evaluate(IChromosome chromosome)
    {
        Evals++;
        return -CountConflicts23(((SudokuR1Chromosome23)chromosome).ToGrid());
    }
}

public static class Mgs23Host
{
    public static (int conflicts, int evals, double ms, double[] genes) RunDe(int seed, int popSize, int maxGens)
    {
        // Seeding AVANT création de population : le RNG est consommé par CreateNew()
        // de chaque individu initial (leçon #12071 / MGS-21).
        FastRandomRandomization.ResetSeed(seed);
        var compound = MetaHeuristicsService.CreateMetaHeuristicByName(
            "DifferentialEvolution", maxGens, popSize);
        var adam = new SudokuR1Chromosome23();
        var pop = new MetaPopulation(popSize, popSize, adam);
        var ga = new MetaGeneticAlgorithm(
            pop, new SudokuR1Fitness23(),
            new EliteSelection(), new UniformCrossover(0.5f), new UniformMutation(true),
            compound);
        ga.Termination = new GenerationNumberTermination(maxGens);
        SudokuR1Fitness23.Evals = 0;
        var sw = Stopwatch.StartNew();
        ga.Start();
        sw.Stop();
        var best = (SudokuR1Chromosome23)ga.BestChromosome;
        return (CountConflicts23(best.ToGrid()), SudokuR1Fitness23.Evals,
                sw.Elapsed.TotalMilliseconds, best.ToGenes());
    }
}

// Échauffement JIT (course jetée), puis course témoin graine 7.
var warmupMgs = Mgs23Host.RunDe(123, 50, 10);
var demoMgs = Mgs23Host.RunDe(7, 50, 160);
Console.WriteLine($"MGS DE (graine 7, témoin) : {demoMgs.Item1} conflits, " +
                  $"{demoMgs.Item2} évaluations, {demoMgs.Item3:F0} ms.");

MGS DE (graine 7, témoin) : 29 conflits, 8000 évaluations, 434 ms.


**Lecture.** Le composé MGS `DifferentialEvolution` (DE/rand/1/bin, F = 0,5, CR = 0,9 par
défaut) est branché sur le même harnais que le PSO de MGS-22 : chromosome R1, fitness comptée,
seeding avant création de population. La course témoin graine 7 donne le premier chiffre — 29 conflits pour 8 000 évaluations en 434 ms —
l'échauffement JIT la précède pour que la course mesurée ne paie pas la compilation.

In [3]:
// === Le pont PythonNet : mealpy dans le même kernel, la même exécution ===
// Recette validée MGS-22 (#12356) : pythonnet 3.1.0, DLL résolue par probe
// (PYTHONNET_PYDLL d'abord, sinon installs standards par OS — aucun chemin machine en dur).
#r "nuget: pythonnet,3.1.0"
using Python.Runtime;
static string ResolvePythonDll23()
{
    var env = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (!string.IsNullOrEmpty(env) && System.IO.File.Exists(env)) return env;
    if (OperatingSystem.IsWindows())
    {
        // Installs CPython.org standards d'abord (les plus récentes portent les packages récents
        // comme mealpy), ensuite le scan LOCALAPPDATA — un Python périmé qui n'a pas mealpy
        // ne doit pas masquer une install plus récente (pb 2026-08-25 : Python310 2023 devant Python313).
        foreach (var c in new[] { @"C:\Python313\python313.dll", @"C:\Python312\python312.dll" })
            if (System.IO.File.Exists(c)) return c;
        var local = Environment.GetEnvironmentVariable("LOCALAPPDATA");
        if (!string.IsNullOrEmpty(local))
        {
            var pyDir = System.IO.Path.Combine(local, "Programs", "Python");
            if (System.IO.Directory.Exists(pyDir))
                foreach (var d in System.IO.Directory.GetDirectories(pyDir, "Python3*"))
                {
                    var hit = System.IO.Directory.GetFiles(d, "python3*.dll");
                    if (hit.Length > 0) return hit[0];
                }
        }
        var home = Environment.GetFolderPath(Environment.SpecialFolder.UserProfile);
        foreach (var root in new[] {
                     System.IO.Path.Combine(home, "miniconda3"),
                     System.IO.Path.Combine(home, "anaconda3"),
                     @"C:\ProgramData\miniconda3",
                     @"C:\ProgramData\anaconda3" })
        {
            if (!System.IO.Directory.Exists(root)) continue;
            var hits = System.IO.Directory.GetFiles(root, "python3*.dll");
            var pick = "";
            foreach (var h in hits)
                if (System.IO.Path.GetFileName(h).Length > 11) pick = h;
            if (pick == "" && hits.Length > 0) pick = hits[0];
            if (pick != "")
            {
                var dirs = new[] { root,
                    System.IO.Path.Combine(root, "Library", "mingw-w64", "bin"),
                    System.IO.Path.Combine(root, "Library", "bin"),
                    System.IO.Path.Combine(root, "Scripts") };
                var path = Environment.GetEnvironmentVariable("PATH") ?? "";
                var toAdd = "";
                foreach (var d in dirs)
                    if (System.IO.Directory.Exists(d) && !path.Contains(d + ";"))
                        toAdd += d + ";";
                if (toAdd != "")
                    Environment.SetEnvironmentVariable("PATH", toAdd + path);
                return pick;
            }
        }
    }
    else
    {
        var libs = new[] { "/usr/lib/x86_64-linux-gnu", "/usr/lib", "/usr/local/lib", "/opt/homebrew/lib" };
        foreach (var dir in libs)
            if (System.IO.Directory.Exists(dir))
            {
                var hit = System.IO.Directory.GetFiles(dir, "libpython3.*");
                foreach (var h in hit)
                    if (h.EndsWith(".so") || h.EndsWith(".dylib")) return h;
            }
    }
    throw new System.IO.FileNotFoundException(
        "DLL Python introuvable : definir PYTHONNET_PYDLL ou installer Python 3.10+ (mealpy requis).");
}
Runtime.PythonDLL = ResolvePythonDll23();
PythonEngine.Initialize();

// Le problème Python : decode + coût réimplémentés à l'identique, compteur d'évals,
// solveur mealpy OriginalDE avec seed EXPLICITE en solve() (API 3.x — le seed du
// constructeur est ignoré) et journal muet (log_to='nothing').
public static PyModule S23;
using (Py.GIL())
{
    S23 = Py.CreateScope();
    S23.Set("puzzle_line23", PuzzleLine23);
    S23.Exec(@"import sys
import mealpy
from mealpy.evolutionary_based.DE import OriginalDE
from mealpy import Problem, FloatVar
import json as _json

puzzle = [int(ch) for ch in puzzle_line23]
empties = [(i // 9, i % 9) for i in range(81) if puzzle[i] == 0]

def decode(vec):
    g = [puzzle[r * 9:(r + 1) * 9] for r in range(9)]
    for k in range(len(empties)):
        r, c = empties[k]
        v = int(round(float(vec[k])))
        g[r][c] = max(1, min(9, v))
    return g

def cost(g):
    conflicts = 0
    for i in range(9):
        units = ([g[i][j] for j in range(9)],
                 [g[j][i] for j in range(9)],
                 [g[3 * (i // 3) + j // 3][3 * (i % 3) + j % 3] for j in range(9)])
        for unit in units:
            seen = set()
            for v in unit:
                if v in seen:
                    conflicts += 1
                seen.add(v)
    return conflicts

def cost_of_vector(vec):
    return cost(decode(vec))

PY_EVALS = [0]

class SudokuProblem(Problem):
    def __init__(self, bounds=None, minmax='min', **kwargs):
        super().__init__(bounds, minmax, log_to='nothing', **kwargs)
    def obj_func(self, x):
        PY_EVALS[0] += 1
        return float(cost(decode(x)))

def run_mealpy_de(seed, pop_size, epoch, wf=None):
    import time
    PY_EVALS[0] = 0
    prob = SudokuProblem(bounds=FloatVar(lb=(1.0,) * len(empties), ub=(10.0,) * len(empties), name='genes'), minmax='min')
    if wf is None:
        model = OriginalDE(epoch=epoch, pop_size=pop_size)
    else:
        model = OriginalDE(epoch=epoch, pop_size=pop_size, wf=wf)
    t0 = time.perf_counter()
    g_best = model.solve(prob, seed=seed)
    dt = (time.perf_counter() - t0) * 1000.0
    sol = _json.dumps([float(v) for v in g_best.solution])
    return cost(decode(g_best.solution)), PY_EVALS[0], dt, sol

def bench_mealpy_de(seeds_json, pop_size, epoch, reps=3, wf=None):
    out = []
    for sd in _json.loads(seeds_json):
        runs = [run_mealpy_de(sd, pop_size, epoch, wf) for _ in range(reps)]
        cs = [r[0] for r in runs]
        es = [r[1] for r in runs]
        ts = sorted(r[2] for r in runs)
        med = ts[len(ts) // 2] if len(ts) % 2 == 1 else (ts[len(ts) // 2 - 1] + ts[len(ts) // 2]) / 2.0
        out.append({'seed': sd, 'conflicts': cs[0], 'all_same': len(set(cs)) == 1,
                    'evals': es[0], 'ms': med, 'sol': runs[0][3]})
    return _json.dumps(out)

def time_python_evals(vecs_json, reps=5):
    import time
    vecs = _json.loads(vecs_json)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        for v in vecs:
            cost_of_vector(v)
        ts.append((time.perf_counter() - t0) * 1000.0)
    ts.sort()
    return ts[len(ts) // 2]

# Defaults mealpy OriginalDE (3.x) : mesures, pas doc
_m = OriginalDE(epoch=10, pop_size=5)
__defaults__ = f'mealpy OriginalDE defaults: wf={_m.wf}, cr={_m.cr}, strategy={_m.strategy}'
__mealpy_ver__ = 'mealpy ' + mealpy.__version__ + ' sur Python ' + sys.version.split()[0]");
    Console.WriteLine($"Pont PythonNet actif : {S23.Get<string>("__mealpy_ver__")}");
    Console.WriteLine($"Parametres defaut mealpy : {S23.Get<string>("__defaults__")}");
}

// --- Sanity check : la fonction de coût est-elle la MÊME des deux côtés ? ---
// 3 vecteurs témoins DÉTERMINISTES (LCG écrit à la main), décodés et costés des deux côtés.
public static double[] LcgVector23(int seed, int n)
{
    uint state = (uint)seed;
    var v = new double[n];
    for (int i = 0; i < n; i++)
    {
        state = state * 1664525u + 1013904223u;
        v[i] = 1.0 + (state / 4294967296.0) * 9.0; // uniforme dans [1, 10)
    }
    return v;
}

var witnessVectors = new[] { LcgVector23(1, 51), LcgVector23(2, 51), LcgVector23(3, 51) };
using (Py.GIL())
{
    S23.Set("__witness_json__",
        System.Text.Json.JsonSerializer.Serialize(witnessVectors.Select(v => v.ToList()).ToList()));
    S23.Exec(@"__py_costs__ = _json.dumps([cost_of_vector(v) for v in _json.loads(__witness_json__)])");
    var pyCosts = System.Text.Json.JsonSerializer.Deserialize<List<int>>(S23.Get<string>("__py_costs__"));
    var csCosts = witnessVectors.Select(v => CountConflicts23(DecodeR1_23(v))).ToList();
    bool identical = pyCosts.SequenceEqual(csCosts);
    Console.WriteLine($"Sanity check cout : C# {string.Join(",", csCosts)} | Python {string.Join(",", pyCosts)} " +
                      $"-> {(identical ? "IDENTIQUE" : "DIFFERENT")}");
}

Installing Packages pythonnet

Pont PythonNet actif : mealpy 3.0.2 sur Python 3.13.3


Parametres defaut mealpy : mealpy OriginalDE defaults: wf=0.1, cr=0.9, strategy=0


Sanity check cout : C# 67,71,60 | Python 67,71,60 -> IDENTIQUE


**Lecture.** Le pont PythonNet est actif et la **sanity check porte tout le bench** : les
trois vecteurs témoins LCG, décodés et coûtés indépendamment des deux côtés, donnent exactement
les mêmes conflits (67, 71, 60 des deux côtés — `IDENTIQUE`). Sans cette égalité prouvée, une différence mesurée entre moteurs pourrait
n'être qu'une différence entre les deux fonctions de coût. Les défauts mealpy (`wf`, `cr`,
`strategy`) sont mesurés sur l'instance, pas recopiés de la doc — c'est la ligne
« Parametres defaut mealpy » ci-dessus qui fait foi pour le confond F déclaré au §1.

In [4]:
// === Moteur mealpy : course témoin + contre-vérification croisée du vainqueur ===
using (Py.GIL())
{
    // Échauffement symétrique (course jetée), puis course témoin graine 7.
    S23.Exec(@"_wu_c, _wu_e, _wu_t, _wu_sol = run_mealpy_de(123, 50, 10)
__d_c__, __d_e__, __d_t__, __d_sol__ = run_mealpy_de(7, 50, 160)");
    int dConflicts = S23.Get<int>("__d_c__");
    int dEvals = S23.Get<int>("__d_e__");
    double dMs = S23.Get<double>("__d_t__");
    Console.WriteLine($"mealpy OriginalDE (graine 7, témoin) : {dConflicts} conflits, " +
                      $"{dEvals} évaluations, {dMs:F0} ms.");

    // Contre-vérification croisée : le vainqueur mealpy, décodé et costé côté C#.
    var solJson = S23.Get<string>("__d_sol__");
    var genes = System.Text.Json.JsonSerializer.Deserialize<double[]>(solJson);
    int csRecheck = CountConflicts23(DecodeR1_23(genes));
    Console.WriteLine($"Contre-vérif croisée : coût C# du meilleur mealpy = {csRecheck} " +
                      $"(Python rapporte {dConflicts}) -> {(csRecheck == dConflicts ? "IDENTIQUE" : "DIFFERENT")}");
}

mealpy OriginalDE (graine 7, témoin) : 20 conflits, 8050 évaluations, 821 ms.


Contre-vérif croisée : coût C# du meilleur mealpy = 20 (Python rapporte 20) -> IDENTIQUE


***

## 2. Le croisement — 2 moteurs × 4 graines à budget égal

Le plan est maintenant le même que MGS-22 : population 50, 160 générations (MGS) / 160 epochs
(mealpy), graines {0, 1, 7, 42}, trois répétitions par graine côté MGS pour la médiane de temps
(amendement anti-pic GC), déterminisme exigé partout.

In [5]:
// === LE BENCH : 2 moteurs x 4 graines {0,1,7,42}, population 50, 160 générations ===
public class BenchRow23
{
    public int seed { get; set; }
    public int conflicts { get; set; }
    public bool all_same { get; set; }
    public int evals { get; set; }
    public double ms { get; set; }
    public string sol { get; set; }
}

int[] Seeds23 = { 0, 1, 7, 42 };

// --- Côté MGS (C#) : 3 répétitions par graine, ms = médiane (amendement §2) ---
var mgsRows = new List<(int seed, int conflicts, int evals, double ms, bool allSame)>();
foreach (var sd in Seeds23)
{
    var runs3 = new List<(int c, int e, double t)>();
    for (int rep = 0; rep < 3; rep++)
    {
        var r = Mgs23Host.RunDe(sd, 50, 160);
        runs3.Add((r.Item1, r.Item2, r.Item3));
    }
    var times = runs3.Select(x => x.t).OrderBy(t => t).ToList();
    double med = times[1];
    mgsRows.Add((sd, runs3[0].c, runs3[0].e, med, runs3.All(x => x.c == runs3[0].c)));
}

// --- Côté mealpy (Python, boucle unique dans le scope) ---
string mealpyJson;
using (Py.GIL())
{
    S23.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(Seeds23.ToList()));
    S23.Exec(@"__bench_json__ = bench_mealpy_de(__seeds_json__, 50, 160)");
    mealpyJson = S23.Get<string>("__bench_json__");
}
var mealpyRows = System.Text.Json.JsonSerializer.Deserialize<List<BenchRow23>>(mealpyJson);

// --- Table ---
static double Median23(List<int> xs)
{
    var s = xs.OrderBy(x => x).ToList();
    return (s.Count % 2 == 1) ? s[s.Count / 2] : (s[s.Count / 2 - 1] + s[s.Count / 2]) / 2.0;
}

Console.WriteLine($"{"moteur",-9} {"graine",6} {"conflits",9} {"evals",7} {"ms",8} {"ms/eval",8}");
foreach (var r in mgsRows)
    Console.WriteLine($"{"MGS",-9} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,8:F0} {r.ms / r.evals,8:F3}");
foreach (var r in mealpyRows)
    Console.WriteLine($"{"mealpy",-9} {r.seed,6} {r.conflicts,9} {r.evals,7} {r.ms,8:F0} {r.ms / r.evals,8:F3}");

var mgsC = mgsRows.Select(r => r.conflicts).ToList();
var mpC = mealpyRows.Select(r => r.conflicts).ToList();
double mgsMsEval = mgsRows.Average(r => r.ms / r.evals);
double mpMsEval = mealpyRows.Average(r => r.ms / r.evals);
Console.WriteLine();
Console.WriteLine($"MGS    : médiane conflits {Median23(mgsC):F1} (min {mgsC.Min()}, max {mgsC.Max()}), ms/éval moyen {mgsMsEval:F3}");
Console.WriteLine($"mealpy : médiane conflits {Median23(mpC):F1} (min {mpC.Min()}, max {mpC.Max()}), ms/éval moyen {mpMsEval:F3}");
Console.WriteLine($"Rapport ms/éval mealpy/MGS : {mpMsEval / mgsMsEval:F2}x");
int detMgs = mgsRows.Count(r => r.allSame) + mealpyRows.Count(r => r.all_same);
Console.WriteLine($"Déterminisme : conflits identiques sur les 3 répétitions pour {detMgs}/8 paires graine-moteur.");

moteur    graine  conflits   evals       ms  ms/eval


MGS            0        20    8000      337    0,042


MGS            1        25    8000      333    0,042


MGS            7        29    8000      331    0,041


MGS           42        26    8000      343    0,043


...

mealpy        42        20    8050      793    0,099


MGS    : médiane conflits 25,5 (min 20, max 29), ms/éval moyen 0,042


mealpy : médiane conflits 21,5 (min 20, max 27), ms/éval moyen 0,100


Rapport ms/éval mealpy/MGS : 2,37x


Déterminisme : conflits identiques sur les 3 répétitions pour 8/8 paires graine-moteur.


In [6]:
// === Coût par évaluation : la fitness seule, hors moteur, 500 vecteurs identiques ===
// Les vecteurs sont générés côté C# (LCG, graines 42..541) et passés en JSON au Python :
// les DEUX côtés chronomètrent decode+coût sur exactement les mêmes 500 points.
int K23 = 500;
var benchVecs = new List<double[]>();
for (int i = 0; i < K23; i++) benchVecs.Add(LcgVector23(42 + i, 51));

var csTimes = new List<double>();
for (int rep = 0; rep < 5; rep++)
{
    var swRep = Stopwatch.StartNew();
    foreach (var v in benchVecs) CountConflicts23(DecodeR1_23(v));
    swRep.Stop();
    csTimes.Add(swRep.Elapsed.TotalMilliseconds);
}
csTimes.Sort();
double csMs = csTimes[2]; // médiane de 5 (amendement §2)

double pyMs;
using (Py.GIL())
{
    S23.Set("__vecs_json__", System.Text.Json.JsonSerializer.Serialize(benchVecs.Select(v => v.ToList()).ToList()));
    S23.Exec(@"__py_ms__ = time_python_evals(__vecs_json__)");
    pyMs = S23.Get<double>("__py_ms__");
}

Console.WriteLine($"Fitness seule, {K23} vecteurs identiques (médiane de 5 répétitions par côté) :");
Console.WriteLine($"  C#     : {csMs:F1} ms total -> {csMs / K23:F3} ms/éval");
Console.WriteLine($"  Python : {pyMs:F1} ms total -> {pyMs / K23:F3} ms/éval");
Console.WriteLine($"  rapport Python/C# : {pyMs / csMs:F2}x");

Fitness seule, 500 vecteurs identiques (médiane de 5 répétitions par côté) :


  C#     : 5,4 ms total -> 0,011 ms/éval


  Python : 20,1 ms total -> 0,040 ms/éval


  rapport Python/C# : 3,71x


**Lecture du croisement.** Les médianes donnent **mealpy devant (21,5 contre 25,5 conflits)**,
mais le tableau change de nature par rapport au PSO de MGS-22 :

- **l'étendue se chevauche** : MGS [20, 29] contre mealpy [20, 27] — et MGS *gagne* la graine 0
  (20 contre 27). La séparation totale du PSO (26-31 contre 39-48) n'est plus là : 3 graines sur 4
  à mealpy, 1 à MGS ;
- **la vitesse s'inverse** : MGS délivre une évaluation 2,37× moins chère (0,042 contre
  0,100 ms/éval ; run original : 3,45×, 0,054 contre 0,188) — avec le PSO, les deux moteurs
  étaient au coude-à-coule (0,81× sur cette machine, 0,99× sur le run original — re-exécutions
  #13407 : les rapports de timing sont sensibles à la machine, l'ordre lui est stable). À budget
  égal, MGS DE termine sa course deux à trois fois plus vite ;
- **le budget est tenu** : 8 000 évaluations MGS contre 8 050 mealpy (la population initiale
  compte pour 50), déterminisme 8/8, contre-vérification croisée du vainqueur IDENTIQUE.

**Lecture du coût par évaluation.** La fitness C# isolée reste 3,71× plus rapide (0,011 contre
0,040 ms/éval ; run original : 6,70× — même ordre de grandeur que le PSO, 5,13× après sa
re-exécution #13407). Mais contrairement au PSO, où la mécanique
moteur mealpy engloutissait l'avantage C#, ici l'écart moteur (2,37×) *reste en dessous* de
l'écart fitness (3,71×) : le composé DE de MGS paie moins de surcoût par évaluation que son PSO.

**Verdict de la paire.** L'écart du PSO ne se reproduit pas tel quel : sur DE, la bibliothèque
distillée MGS rejoint mealpy en qualité sur un quart des graines et domine nettement en temps par
évaluation, sans refermer entièrement l'écart médian. Le confond F (0,5 contre 0,1) reste la
première explication à écarter — c'est l'exercice 1.

***

## Exercice 1 : neutraliser le confond F — mealpy `wf` aligné sur MGS

Le protocole compare les bibliothèques **à leurs paramètres par défaut**, et les défauts
diffèrent : MGS prend F = 0,5, mealpy prend `wf` = 0,1. Si l'écart du croisement tenait surtout
à ce paramètre, l'aligner devrait le refermer en grande partie.

```text
À compléter (décommentez dans la cellule suivante) :
1. Relancez bench_mealpy_de avec wf=0.5 (l'argument optionnel est déjà câblé dans run_mealpy_de).
2. Comparez la médiane obtenue à la médiane MGS du croisement.
3. Verdict : l'écart survit-il à l'alignement du facteur d'échelle ?
```

In [7]:
// EXERCICE 1 : mealpy OriginalDE avec wf=0.5 (aligné sur le DefaultScaleFactor MGS),
// 4 graines, mêmes budget/population que le croisement.
// Décommentez et exécutez :
// using (Py.GIL())
// {
//     S23.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(new[] {0, 1, 7, 42}.ToList()));
//     S23.Exec(@"__bench_wf_json__ = bench_mealpy_de(__seeds_json__, 50, 160, wf=0.5)");
//     Console.WriteLine(S23.Get<string>("__bench_wf_json__"));
// }

// Indice : la fonction Python accepte déjà wf optionnel — mesurez avant de conclure.

Console.WriteLine("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 2 : budget ×4 — l'écart de qualité se referme-t-il ?

MGS-22 posait la même question pour le PSO. Un écart qui se referme à budget accru dit « le
moteur distillé converge plus lentement mais atteint le même plateau » ; un écart stable dit
« plateau différent ».

In [8]:
// EXERCICE 2 : budget x4 (pop 50, 640 générations/epochs), 4 graines, deux côtés.
// Décommentez et exécutez :
// foreach (var sd in new[] {0, 1, 7, 42})
// {
//     var r = Mgs23Host.RunDe(sd, 50, 640);
//     Console.WriteLine($"MGS DE x4 (graine {sd}) : {r.Item1} conflits, {r.Item2} évaluations, {r.Item3:F0} ms.");
// }
// using (Py.GIL())
// {
//     S23.Set("__seeds_json__", System.Text.Json.JsonSerializer.Serialize(new[] {0, 1, 7, 42}.ToList()));
//     S23.Exec(@"__bench_x4_json__ = bench_mealpy_de(__seeds_json__, 50, 640)");
//     Console.WriteLine(S23.Get<string>("__bench_x4_json__"));
// }

Console.WriteLine("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).


## Exercice 3 : profiler la fitness — où va la milliseconde ?

La cellule du coût par évaluation compare déjà le total decode+coût. Pour localiser la
différence, séparez les deux étapes côté Python (dé coder une fois, coûter N fois) et comparez
au profil C# équivalent.

In [9]:
// EXERCICE 3 : profil decode vs cost, 500 vecteurs, deux côtés.
// Décommentez et exécutez (adapté de MGS-22 exercice 3) :
// var decSw = Stopwatch.StartNew();
// foreach (var v in benchVecs) DecodeR1_23(v);
// decSw.Stop();
// Console.WriteLine($"C# decode seul : {decSw.Elapsed.TotalMilliseconds / K23:F3} ms/vec " +
//     $"(reste = coût : {(csMs - decSw.Elapsed.TotalMilliseconds) / K23:F3} ms/vec)");
// using (Py.GIL())
// {
//     S23.Set("__vecs_json__", System.Text.Json.JsonSerializer.Serialize(benchVecs.Select(v => v.ToList()).ToList()));
//     S23.Exec(@"import time
// _vecs = _json.loads(__vecs_json__)
// _t0 = time.perf_counter()
// _grids = [decode(v) for v in _vecs]
// _t1 = time.perf_counter()
// for g in _grids: cost(g)
// _t2 = time.perf_counter()
// print(f'Python decode seul : {(_t1-_t0)*1000.0/len(_vecs):.3f} ms/vec, coût : {(_t2-_t1)*1000.0/len(_vecs):.3f} ms/vec')");
// }

Console.WriteLine("Exercice a completer (decommentez le bloc ci-dessus).");

Exercice a completer (decommentez le bloc ci-dessus).
